In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip install transformers accelerate sacremoses sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 16.8 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import random
from collections import Counter
from tqdm.auto import tqdm

In [ ]:
# import gdown
# import zipfile
# import os
# import torch
# from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# MODEL_INFO = {
#     "mbart_ft": {"id": "1O6G2SH9ri1nb6w4YR_263uJ_0p8do6T2", "archive": "mbart.zip", "folder": "mbart_fine_tuned_essential2"},
#     "nllb_ft": {"id": "1uEySNZMYAylShxVxrDuhjlKavXlx_9P8", "archive": "nllb.zip", "folder": "fine_tuned_nllb_tourism_essential2"},
#     "helsinki_ft": {"id": "1rPNS5q2BeurL6yJIV1YZobR_QlayQtbj", "archive": "helsinki.zip", "folder": "fine_tuned_helsinki_tourism2"},

#     "custom_pt": {"id": "1_KkZH_E4xKg7n_DY56lP13qJPj9roteg", "filename": "LSTM2.pt"},
# }

# # Define a directory for all downloads
# DOWNLOAD_DIR = "./downloaded_models"
# os.makedirs(DOWNLOAD_DIR, exist_ok=True)

In [ ]:
# loaded_hf_models = {}

# print("--- Downloading and Extracting Hugging Face Models ---")
# for key, info in MODEL_INFO.items():
#     if 'archive' in info:
#         archive_path = os.path.join(DOWNLOAD_DIR, info['archive'])

#         # Download
#         print(f"Downloading {key}...")
#         gdown.download(id=info['id'], output=archive_path, quiet=True)

#         # Extract
#         with zipfile.ZipFile(archive_path, 'r') as zip_ref:
#             # Extract to the download directory
#             zip_ref.extractall(DOWNLOAD_DIR)

#         # Load Model and Tokenizer
#         model_path = os.path.join(DOWNLOAD_DIR, info['folder'])
#         print(f"Loading model from {model_path}...")

#         loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)
#         loaded_model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

#         loaded_hf_models[key] = {'model': loaded_model, 'tokenizer': loaded_tokenizer}
#         print(f"Loaded {key.upper()} successfully.")

# # Access your models, e.g., loaded_hf_models['mbart_ft']['model']

In [ ]:
test_df = pd.read_csv("https://drive.google.com/uc?id=1eX6RCAxYS8g_DAOgg4pvIEqzph6Y15kU")
random_rows = test_df['text'].sample(5)

# LSTM

## Declare

In [ ]:
class Vocabulary:
    def __init__(self):
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.freq_threshold = 1

    def __len__(self):
        return len(self.stoi)

    @staticmethod
    def tokenizer(text):
        return text.lower().split()

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4

        for sentence in sentence_list:
            for word in self.tokenizer(sentence):
                frequencies[word] += 1

                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        tokenized_text = self.tokenizer(text)
        return [
            self.stoi[token] if token in self.stoi else self.stoi["<UNK>"]
            for token in tokenized_text
        ]

    class ParallelDataset(Dataset):
    def __init__(self, df, src_col, trg_col, src_vocab, trg_vocab):
        self.df = df
        self.src_col = src_col
        self.trg_col = trg_col
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        src_text = self.df.iloc[index][self.src_col]
        trg_text = self.df.iloc[index][self.trg_col]

        # Convert text to indices
        src_indices = [self.src_vocab.stoi["<SOS>"]] + \
                      self.src_vocab.numericalize(src_text) + \
                      [self.src_vocab.stoi["<EOS>"]]

        trg_indices = [self.trg_vocab.stoi["<SOS>"]] + \
                      self.trg_vocab.numericalize(trg_text) + \
                      [self.trg_vocab.stoi["<EOS>"]]

        return torch.tensor(src_indices), torch.tensor(trg_indices)

# Collate function to pad batches to the same length
class MyCollate:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        src = [item[0] for item in batch]
        trg = [item[1] for item in batch]

        # Pad sequences
        src = torch.nn.utils.rnn.pad_sequence(src, batch_first=True, padding_value=self.pad_idx)
        trg = torch.nn.utils.rnn.pad_sequence(trg, batch_first=True, padding_value=self.pad_idx)

        return src, trg
def load_pretrained_embeddings(vocab, file_path, emb_dim=300):
    """
    Args:
        vocab: Your Vocabulary object (src_vocab or trg_vocab)
        file_path: Path to the .vec file (e.g., 'wiki.id.vec')
        emb_dim: Dimension of the vectors (FastText is usually 300)
    Returns:
        FloatTensor of shape (vocab_size, emb_dim)
    """
    print(f"Loading embeddings from {file_path}...")

    vocab_size = len(vocab)
    embedding_matrix = torch.randn(vocab_size, emb_dim)

    hit_count = 0
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        next(f)

        for line in tqdm(f, desc="Parsing Vectors"):
            parts = line.split()
            word = parts[0]

            if word in vocab.stoi:
                idx = vocab.stoi[word]
                try:
                    vector = torch.tensor([float(x) for x in parts[1:]])
                    if vector.shape[0] == emb_dim:
                        embedding_matrix[idx] = vector
                        hit_count += 1
                except:
                    continue

    print(f"Found {hit_count} / {vocab_size} words in pretrained file.")
    print(f"Coverage: {hit_count / vocab_size * 100:.2f}%")

    return embedding_matrix
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout, pretrained_weights=None):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_dim, emb_dim)

        if pretrained_weights is not None:
            self.embedding.weight.data.copy_(pretrained_weights)
            self.embedding.weight.requires_grad = True

        # 1. Bidirectional
        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers,
                            dropout=dropout, batch_first=True,
                            bidirectional=True)

        self.dropout = nn.Dropout(dropout)

        self.fc_hidden = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_cell = nn.Linear(hid_dim * 2, hid_dim)

    def forward(self, src):
        # src = [batch size, src len]
        embedded = self.dropout(self.embedding(src))

        # outputs = [batch size, src len, hid_dim * 2]
        # hidden = [n_layers * 2, batch size, hid_dim]
        # cell = [n_layers * 2, batch size, hid_dim]
        outputs, (hidden, cell) = self.lstm(embedded)

        # 3. Handle the Hidden/Cell States
        # We need to concatenate the forward and backward states of the LAST layer
        # hidden[-2, :, :] is the Forward state of the last layer
        # hidden[-1, :, :] is the Backward state of the last layer

        # Concatenate forward + backward
        hidden_cat = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        cell_cat = torch.cat((cell[-2,:,:], cell[-1,:,:]), dim=1)

        # Squash them through the linear layer to match Decoder size
        # shape becomes [batch size, hid_dim]
        hidden = torch.tanh(self.fc_hidden(hidden_cat))
        cell = torch.tanh(self.fc_cell(cell_cat))

        # 4. Prepare for Decoder
        # The decoder expects shape [n_layers, batch, hid_dim].
        # Since we squashed it to [batch, hid_dim], we typically unsqueeze
        # to fit n_layers=1 for the decoder.
        # (Note: This simple implementation assumes Decoder n_layers=1)
        return hidden.unsqueeze(0), cell.unsqueeze(0)

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout, pretrained_weights=None):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)

        if pretrained_weights is not None:
            self.embedding.weight.data.copy_(pretrained_weights)
            self.embedding.weight.requires_grad = False

        # Decoder remains UNIDIRECTIONAL
        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)

        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))

        # output = [batch size, 1, hid_dim]
        # hidden = [n_layers, batch, hid_dim]
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))

        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        # Encoder
        hidden, cell = self.encoder(src)

        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1

        return outputs
def predict_sentence(model, sentence, src_vocab, trg_vocab, device, max_len=50):
    model.eval()

    # Numericalize
    if isinstance(sentence, str):
        tokens = src_vocab.numericalize(sentence)
    else:
        return ""

    tokens = [src_vocab.stoi["<SOS>"]] + tokens + [src_vocab.stoi["<EOS>"]]
    src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device) # [1, seq_len]

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    trg_indexes = [trg_vocab.stoi["<SOS>"]]

    # Decode step by step
    for _ in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)

        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)

        if pred_token == trg_vocab.stoi["<EOS>"]:
            break

    # Convert back to words
    trg_tokens = [trg_vocab.itos[i] for i in trg_indexes]

    # Remove SOS and EOS for cleaner output
    return " ".join(trg_tokens[1:-1])
import pandas as pd

src_vocab = Vocabulary()
trg_vocab = Vocabulary()

print("Building vocabulary from Training data...")
train_df = pd.read_csv("https://drive.google.com/uc?id=12l_YUBjcAT3CnEYCqmMQwFwTj9ft5N7M")
src_vocab.build_vocabulary(train_df['text'].tolist())
trg_vocab.build_vocabulary(train_df['english_translation'].tolist())

print(f"Source Vocab Size: {len(src_vocab)}")
print(f"Target Vocab Size: {len(trg_vocab)}")
ENC_EMB_DIM = 300
DEC_EMB_DIM = 300
HID_DIM = 256
N_LAYERS = 1
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

In [ ]:
# pt_info = MODEL_INFO["custom_pt"]
# pt_path = os.path.join(DOWNLOAD_DIR, pt_info['filename'])

# # Download the .pt file
# print(f"Downloading {pt_info['filename']}...")
# gdown.download(id=pt_info['id'], output=pt_path, quiet=True)

# 2. Instantiate Model
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder = Encoder(len(src_vocab), ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
decoder = Decoder(len(trg_vocab), DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

# 2. Load the .pt file
MODEL_PATH = '/content/drive/MyDrive/Model/LSTM2.pt'
state_dict = torch.load(MODEL_PATH, map_location=DEVICE)

# 3. Apply the weights
# This loads the weights (state_dict) into the model's architecture
model.load_state_dict(state_dict)
model.eval() # Set model to evaluation mode

print("Model successfully loaded and ready for prediction.")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn(


Model successfully loaded and ready for prediction.


## Inference

In [ ]:
# text_to_translate = "Apakah ada restoran yang menyajikan makanan halal di dekat hotel ini?"
for text in random_rows:
    text_to_translate = text
    translation = predict_sentence(
        model,
        text_to_translate,
        src_vocab,
        trg_vocab,
        DEVICE,
        max_len=50 # Use a sensible max_len to prevent infinite loops
    )

    # 3. Print the result
    print(f"Original Text: {text_to_translate}")
    print(f"Translation: {translation}")

Original Text: kalau mau datang kesini mending musim kemarau biar dapat air yang hijau jernih kalau hujan keruh airnya coklat
Translation: the place is is the the the the the the the the the the the the the the the the the the the the the the
Original Text: tempat wisata yang menarik untuk dikunjungi oleh peminat yang alami sejuk dan nyaman
Translation: the place is is the the the the the the the the the the the the the the the the the the the the the the
Original Text: recommended pisan ini kita saja sudah berpa kali ke sini enggak ada bosennya kumplit mencari makan rupa aneka pilihan main anak anak asa jalan jalan keliling doang apalagi mau main air banyak asik fresh lah kalo pulang
Translation: the place is is the the the the the the the the the the the the the the the the the the the the the the
Original Text: masyaallah bagus khusus yang kuat naik turun tangga hati-hati ada monyet
Translation: the place is is the the the the the the the the the the the the the the the the the the 

# Opus-mt-id-en

In [3]:
# loaded_tokenizer = loaded_hf_models['helsinki_ft']['tokenizer']
# loaded_model = loaded_hf_models['helsinki_ft']['model']

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

LOAD_DIR = "Helsinki-NLP/opus-mt-id-en"

loaded_tokenizer = AutoTokenizer.from_pretrained(LOAD_DIR)
loaded_model = AutoModelForSeq2SeqLM.from_pretrained(LOAD_DIR)

print(f"Model and tokenizer successfully loaded from {LOAD_DIR}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/801k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/796k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/291M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Model and tokenizer successfully loaded from Helsinki-NLP/opus-mt-id-en


In [5]:
# Sample text for translation
text_to_translate = "depannya sih lumayan bagus banyak bunganya dan spotnya juga bagus tapi pas di kebun sayur kok kayak enggak "
input_ids = loaded_tokenizer(
    text_to_translate,
    return_tensors="pt",
    max_length=128,
    truncation=True
).input_ids.to(loaded_model.device)

outputs = loaded_model.generate(input_ids)

translation = loaded_tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Original Text: {text_to_translate}")
print(f"Translation: {translation}")

Original Text: depannya sih lumayan bagus banyak bunganya dan spotnya juga bagus tapi pas di kebun sayur kok kayak enggak 
Translation: fontcolor=" # FFFF00"the frontis prettygood fontcolor=" # FFFF00"manyflowersandthespotisgood fontcolor=" # FFFF00"butrightin the vegetableplant fontcolor=" # FFFF00"likeno


# NLLB

In [ ]:
# loaded_tokenizer = loaded_hf_models['nllb_ft']['tokenizer']
# loaded_model = loaded_hf_models['nllb_ft']['model']

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

LOAD_DIR = "/content/drive/MyDrive/Model/fine_tuned_nllb_tourism_essential2"

loaded_tokenizer = AutoTokenizer.from_pretrained(LOAD_DIR)
loaded_model = AutoModelForSeq2SeqLM.from_pretrained(LOAD_DIR)

print(f"Model and tokenizer successfully loaded from {LOAD_DIR}")

Model and tokenizer successfully loaded from /content/drive/MyDrive/Model/fine_tuned_nllb_tourism_essential2


In [ ]:
text_to_translate = "Apakah ada restoran yang menyajikan makanan halal di dekat hotel ini?"
for text in random_rows:
    text_to_translate = text
    TARGET_LANG_CODE = "eng_Latn"

    input_ids = loaded_tokenizer(
        text_to_translate,
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).input_ids.to(loaded_model.device)

    outputs = loaded_model.generate(
        input_ids,
        forced_bos_token_id = loaded_tokenizer.convert_tokens_to_ids(TARGET_LANG_CODE)
    )

    translation = loaded_tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"Original Text: {text_to_translate}")
    print(f"Target Language Code: {TARGET_LANG_CODE}")
    print(f"Translation: {translation}")

Original Text: kalau mau datang kesini mending musim kemarau biar dapat air yang hijau jernih kalau hujan keruh airnya coklat
Target Language Code: eng_Latn
Translation: ?If you want to come here, it's better during the dry season to get clear green water. If it rains, the water will be brown.
Original Text: tempat wisata yang menarik untuk dikunjungi oleh peminat yang alami sejuk dan nyaman
Target Language Code: eng_Latn
Translation: ?An interesting tourist spot to visit by nature lovers, cool and comfortable.
Original Text: recommended pisan ini kita saja sudah berpa kali ke sini enggak ada bosennya kumplit mencari makan rupa aneka pilihan main anak anak asa jalan jalan keliling doang apalagi mau main air banyak asik fresh lah kalo pulang
Target Language Code: eng_Latn
Translation: ? Recommended. We've been here several times. There's no excuse. I'm looking for food. There's a variety of choices. Kids don't want to walk around, especially if they want to play in the water. It's fun a

# mBART

In [ ]:
# loaded_tokenizer = loaded_hf_models['mbart_ft_ft']['tokenizer']
# loaded_model = loaded_hf_models['mbart_ft_ft']['model']

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

LOAD_DIR = "/content/drive/MyDrive/Model/mbart_fine_tuned_essential2"

loaded_tokenizer = AutoTokenizer.from_pretrained(LOAD_DIR)
loaded_model = AutoModelForSeq2SeqLM.from_pretrained(LOAD_DIR)

print(f"Model and tokenizer successfully loaded from {LOAD_DIR}")

Model and tokenizer successfully loaded from /content/drive/MyDrive/Model/mbart_fine_tuned_essential2


In [ ]:
text_to_translate = "Apakah ada restoran yang menyajikan makanan halal di dekat hotel ini?"
for text in random_rows:
    text_to_translate = text
    TARGET_LANG_CODE = "en_XX"

    input_ids = loaded_tokenizer(
        text_to_translate,
        return_tensors="pt",
        max_length=128,
        truncation=True
    ).input_ids.to(loaded_model.device)

    outputs = loaded_model.generate(
        input_ids,
        forced_bos_token_id=loaded_tokenizer.lang_code_to_id[TARGET_LANG_CODE]
    )

    translation = loaded_tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"Original Text: {text_to_translate}")
    print(f"Target Language Code: {TARGET_LANG_CODE}")
    print(f"Translation: {translation}")

Original Text: Apakah ada restoran yang menyajikan makanan halal di dekat hotel ini?
Target Language Code: en_XX
Translation: Is there a restaurant serving halal food near this hotel?


# Testing

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import random
import os
from collections import Counter
from typing import List, Optional, Union, Dict, Any
from abc import ABC, abstractmethod
from tqdm import tqdm


# Base Abstract Class
class BaseTranslator(ABC):
    """Abstract base class for all translation models."""

    def __init__(self, device: Optional[str] = None):
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None
        self.is_loaded = False

    @abstractmethod
    def load_model(self, model_path: str):
        """Load the translation model."""
        pass

    @abstractmethod
    def translate_text(self, text: str, **kwargs) -> str:
        """Translate a single text."""
        pass

    @abstractmethod
    def translate_batch(self, texts: List[str], **kwargs) -> List[str]:
        """Translate a batch of texts."""
        pass

    def get_model_info(self) -> dict:
        """Get basic model information."""
        return {
            "model_type": self.__class__.__name__,
            "device": self.device,
            "is_loaded": self.is_loaded
        }


# LSTM Model Components
class Vocabulary:
    def __init__(self):
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.freq_threshold = 1

    def __len__(self):
        return len(self.stoi)

    @staticmethod
    def tokenizer(text):
        return text.lower().split()

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4

        for sentence in sentence_list:
            for word in self.tokenizer(sentence):
                frequencies[word] += 1

                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        tokenized_text = self.tokenizer(text)
        return [
            self.stoi[token] if token in self.stoi else self.stoi["<UNK>"]
            for token in tokenized_text
        ]


class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout, pretrained_weights=None):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_dim, emb_dim)

        if pretrained_weights is not None:
            self.embedding.weight.data.copy_(pretrained_weights)
            self.embedding.weight.requires_grad = True

        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers,
                            dropout=dropout, batch_first=True,
                            bidirectional=True)

        self.dropout = nn.Dropout(dropout)
        self.fc_hidden = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_cell = nn.Linear(hid_dim * 2, hid_dim)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.lstm(embedded)

        hidden_cat = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        cell_cat = torch.cat((cell[-2,:,:], cell[-1,:,:]), dim=1)

        hidden = torch.tanh(self.fc_hidden(hidden_cat))
        cell = torch.tanh(self.fc_cell(cell_cat))

        return hidden.unsqueeze(0), cell.unsqueeze(0)


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout, pretrained_weights=None):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)

        if pretrained_weights is not None:
            self.embedding.weight.data.copy_(pretrained_weights)
            self.embedding.weight.requires_grad = False

        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)
        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1

        return outputs


# Individual Translator Classes
class LSTMTranslator(BaseTranslator):
    """Custom LSTM-based translator."""

    def __init__(self, device: Optional[str] = None, config: Optional[Dict] = None):
        super().__init__(device)
        self.config = config or self._get_default_config()
        self.src_vocab = None
        self.trg_vocab = None

    def _get_default_config(self):
        return {
            'ENC_EMB_DIM': 300,
            'DEC_EMB_DIM': 300,
            'HID_DIM': 256,
            'N_LAYERS': 1,
            'ENC_DROPOUT': 0.5,
            'DEC_DROPOUT': 0.5,
            'vocab_data_url': "https://drive.google.com/uc?id=12l_YUBjcAT3CnEYCqmMQwFwTj9ft5N7M"
        }

    def load_model(self, model_path: str):
        """Load LSTM model and build vocabularies."""
        try:
            self.src_vocab = Vocabulary()
            self.trg_vocab = Vocabulary()

            # Load training data for vocabulary building
            train_df = pd.read_csv(self.config['vocab_data_url'])
            self.src_vocab.build_vocabulary(train_df['text'].tolist())
            self.trg_vocab.build_vocabulary(train_df['english_translation'].tolist())

            # Initialize model architecture
            encoder = Encoder(
                len(self.src_vocab),
                self.config['ENC_EMB_DIM'],
                self.config['HID_DIM'],
                self.config['N_LAYERS'],
                self.config['ENC_DROPOUT']
            )
            decoder = Decoder(
                len(self.trg_vocab),
                self.config['DEC_EMB_DIM'],
                self.config['HID_DIM'],
                self.config['N_LAYERS'],
                self.config['DEC_DROPOUT']
            )
            self.model = Seq2Seq(encoder, decoder, self.device).to(self.device)

            # Load pre-trained weights
            if os.path.exists(model_path):
                state_dict = torch.load(model_path, map_location=self.device)
                self.model.load_state_dict(state_dict)
                self.model.eval()

            self.is_loaded = True

        except Exception as e:
            raise e

    def translate_text(self, text: str, max_length: int = 50, **kwargs) -> str:
        """Translate text using LSTM model."""
        if not self.is_loaded:
            raise RuntimeError("Model not loaded. Please call load_model() first.")

        self.model.eval()

        if isinstance(text, str):
            tokens = self.src_vocab.numericalize(text)
        else:
            return ""

        tokens = [self.src_vocab.stoi["<SOS>"]] + tokens + [self.src_vocab.stoi["<EOS>"]]
        src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(self.device)

        with torch.no_grad():
            hidden, cell = self.model.encoder(src_tensor)

        trg_indexes = [self.trg_vocab.stoi["<SOS>"]]

        for _ in range(max_length):
            trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(self.device)
            with torch.no_grad():
                output, hidden, cell = self.model.decoder(trg_tensor, hidden, cell)

            pred_token = output.argmax(1).item()
            trg_indexes.append(pred_token)

            if pred_token == self.trg_vocab.stoi["<EOS>"]:
                break

        trg_tokens = [self.trg_vocab.itos[i] for i in trg_indexes]
        return " ".join(trg_tokens[1:-1])

    def translate_batch(self, texts: List[str], **kwargs) -> List[str]:
        """Translate batch of texts using LSTM model."""
        return [self.translate_text(text, **kwargs) for text in texts]


class HelsinkiTranslator(BaseTranslator):
    """Helsinki NLP translator."""

    def __init__(self, device: Optional[str] = None):
        super().__init__(device)
        self.tokenizer = None

    def load_model(self, model_path: str):
        """Load Helsinki NLP model from local path or HuggingFace Hub."""
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_path)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
            self.model.to(self.device)

            self.is_loaded = True

        except Exception as e:
            raise e

    def translate_text(self, text: str, max_length: int = 128, **kwargs) -> str:
        """Translate text using Helsinki model."""
        if not self.is_loaded:
            raise RuntimeError("Model not loaded. Please call load_model() first.")

        try:
            input_ids = self.tokenizer(
                text,
                return_tensors="pt",
                max_length=128,
                truncation=True
            ).input_ids.to(self.device)

            outputs = self.model.generate(
                input_ids,
                max_length=max_length,
                do_sample=False
            )

            translation = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            return translation
        except Exception as e:
            return ""

    def translate_batch(self, texts: List[str], max_length: int = 128, batch_size: int = 8, **kwargs) -> List[str]:
        """Translate batch of texts using Helsinki model."""
        if not self.is_loaded:
            raise RuntimeError("Model not loaded. Please call load_model() first.")

        translations = []

        try:
            for text in texts:
                translation = self.translate_text(text, max_length=max_length)
                translations.append(translation)
        except Exception as e:
            pass

        return translations


class NLLBTranslator(BaseTranslator):
    """NLLB (No Language Left Behind) translator."""

    def __init__(self, device: Optional[str] = None):
        super().__init__(device)
        self.tokenizer = None

    def load_model(self, model_path: str):
        """Load NLLB model from local path or HuggingFace Hub."""
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_path)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
            self.model.to(self.device)

            self.is_loaded = True

        except Exception as e:
            raise e

    def translate_text(self, text: str, max_length: int = 128, src_lang: str = "ind_Latn", tgt_lang: str = "eng_Latn", **kwargs) -> str:
        """Translate text using NLLB model. Defaults to Indonesian->English."""
        if not self.is_loaded:
            raise RuntimeError("Model not loaded. Please call load_model() first.")

        try:
            input_ids = self.tokenizer(
                text,
                return_tensors="pt",
                max_length=128,
                truncation=True
            ).input_ids.to(self.device)

            outputs = self.model.generate(
                input_ids,
                forced_bos_token_id=self.tokenizer.convert_tokens_to_ids(tgt_lang),
                max_length=max_length,
                do_sample=False
            )

            translation = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            return translation
        except Exception as e:
            return ""

    def translate_batch(self, texts: List[str], max_length: int = 128, batch_size: int = 8, **kwargs) -> List[str]:
        """Translate batch of texts using NLLB model."""
        return [self.translate_text(text, max_length=max_length, **kwargs) for text in texts]


class MBartTranslator(BaseTranslator):
    """mBART (Multilingual BART) translator."""

    def __init__(self, device: Optional[str] = None):
        super().__init__(device)
        self.tokenizer = None

    def load_model(self, model_path: str):
        """Load mBART model from local path or HuggingFace Hub."""
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_path)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
            self.model.to(self.device)

            self.is_loaded = True

        except Exception as e:
            raise e

    def translate_text(self, text: str, max_length: int = 128, src_lang: str = "id_ID", tgt_lang: str = "en_XX", **kwargs) -> str:
        """Translate text using mBART model. Defaults to Indonesian->English."""
        if not self.is_loaded:
            raise RuntimeError("Model not loaded. Please call load_model() first.")

        try:
            # Handle different language code formats
            if hasattr(self.tokenizer, 'src_lang'):
                self.tokenizer.src_lang = src_lang

            input_ids = self.tokenizer(
                text,
                return_tensors="pt",
                max_length=128,
                truncation=True
            ).input_ids.to(self.device)

            # Try different ways to get language token ID
            forced_bos_token_id = None
            if hasattr(self.tokenizer, 'lang_code_to_id') and tgt_lang in self.tokenizer.lang_code_to_id:
                forced_bos_token_id = self.tokenizer.lang_code_to_id[tgt_lang]
            elif hasattr(self.tokenizer, 'convert_tokens_to_ids'):
                try:
                    forced_bos_token_id = self.tokenizer.convert_tokens_to_ids(tgt_lang)
                except:
                    pass

            if forced_bos_token_id:
                outputs = self.model.generate(
                    input_ids,
                    forced_bos_token_id=forced_bos_token_id,
                    max_length=max_length,
                    do_sample=False
                )
            else:
                outputs = self.model.generate(
                    input_ids,
                    max_length=max_length,
                    do_sample=False
                )

            translation = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            return translation
        except Exception as e:
            return ""

    def translate_batch(self, texts: List[str], max_length: int = 128, batch_size: int = 8, **kwargs) -> List[str]:
        """Translate batch of texts using mBART model."""
        return [self.translate_text(text, max_length=max_length, **kwargs) for text in texts]


# Main Translation Class
class Translation:
    """Main Translation class that manages different translation models."""

    def __init__(self, device: Optional[str] = None):
        """
        Initialize the Translation class.

        Args:
            device (str, optional): Device to run models on ('cpu' or 'cuda').
        """
        self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")
        self.translators = {}

    def load_lstm_model(self, model_path: str = "'/content/drive/MyDrive/Model/LSTM2.pt", config: Optional[Dict] = None):
        """Load LSTM translation model."""
        self.translators['lstm'] = LSTMTranslator(self.device, config)
        self.translators['lstm'].load_model(model_path)

    def load_helsinki_model(self, model_name: str = "/content/drive/MyDrive/Model/fine_tuned_helsinki_tourism2"):
        """Load Helsinki NLP translation model."""
        self.translators['helsinki'] = HelsinkiTranslator(self.device)
        self.translators['helsinki'].load_model(model_name)

    def load_nllb_model(self, model_name: str = "/content/drive/MyDrive/Model/fine_tuned_nllb_tourism_essential2"):
        """Load NLLB translation model."""
        self.translators['nllb'] = NLLBTranslator(self.device)
        self.translators['nllb'].load_model(model_name)

    def load_mbart_model(self, model_name: str = "/content/drive/MyDrive/Model/mbart_fine_tuned_essential2"):
        """Load mBART translation model."""
        self.translators['mbart'] = MBartTranslator(self.device)
        self.translators['mbart'].load_model(model_name)

    def translate_text(self, text: str, model_type: str, **kwargs) -> str:
        """
        Translate text using specified model.

        Args:
            text (str): Text to translate
            model_type (str): Type of model ('lstm', 'helsinki', 'nllb', 'mbart')
            **kwargs: Additional arguments for translation

        Returns:
            str: Translated text
        """
        if model_type not in self.translators:
            raise ValueError(f"Model '{model_type}' not loaded. Available models: {list(self.translators.keys())}")

        return self.translators[model_type].translate_text(text, **kwargs)

    def translate_batch(self, texts: List[str], model_type: str, **kwargs) -> List[str]:
        """
        Translate batch of texts using specified model.

        Args:
            texts (List[str]): List of texts to translate
            model_type (str): Type of model ('lstm', 'helsinki', 'nllb', 'mbart')
            **kwargs: Additional arguments for translation

        Returns:
            List[str]: List of translated texts
        """
        if model_type not in self.translators:
            raise ValueError(f"Model '{model_type}' not loaded. Available models: {list(self.translators.keys())}")

        return self.translators[model_type].translate_batch(texts, **kwargs)

    def get_available_models(self) -> List[str]:
        """Get list of loaded models."""
        return list(self.translators.keys())

    def get_model_info(self, model_type: str = None) -> Dict:
        """
        Get information about loaded models.

        Args:
            model_type (str, optional): Specific model to get info for. If None, returns all models info.

        Returns:
            dict: Model information
        """
        if model_type:
            if model_type not in self.translators:
                raise ValueError(f"Model '{model_type}' not loaded.")
            return self.translators[model_type].get_model_info()
        else:
            return {model_type: translator.get_model_info()
                    for model_type, translator in self.translators.items()}

In [ ]:

# 1. Initialize the main translation class
translator = Translation()

# 2. Load your fine-tuned models from Google Drive (mounted as /content/drive)
print("Loading models...")

# Load LSTM model (your custom trained model)
translator.load_lstm_model()

# Load Helsinki model (fine-tuned or standard)
translator.load_helsinki_model()
# Or use standard model: translator.load_helsinki_model("Helsinki-NLP/opus-mt-id-en")

# Load NLLB model (your fine-tuned version)
translator.load_nllb_model()

# Load mBART model (your fine-tuned version)
translator.load_mbart_model()

print("All models loaded successfully!")

# 3. Check which models are available
available_models = translator.get_available_models()
print(f"Available models: {available_models}")

# 4. Single text translation examples
indonesian_text = "Saya sangat senang belajar kecerdasan buatan di universitas."

print("\n=== Single Text Translation ===")

# Translate using LSTM (defaults to Indonesian -> English)
if 'lstm' in available_models:
    lstm_result = translator.translate_text(indonesian_text, "lstm")
    print(f"LSTM: {lstm_result}")

# Translate using Helsinki
if 'helsinki' in available_models:
    helsinki_result = translator.translate_text(indonesian_text, "helsinki")
    print(f"Helsinki: {helsinki_result}")

# Translate using NLLB (defaults to ind_Latn -> eng_Latn)
if 'nllb' in available_models:
    nllb_result = translator.translate_text(indonesian_text, "nllb")
    print(f"NLLB: {nllb_result}")

# Translate using mBART (defaults to id_ID -> en_XX)
if 'mbart' in available_models:
    mbart_result = translator.translate_text(indonesian_text, "mbart")
    print(f"mBART: {mbart_result}")

# 5. Batch translation example
print("\n=== Batch Translation ===")

indonesian_sentences = [
    "Selamat pagi, bagaimana kabarmu?",
    "Cuaca hari ini sangat cerah dan indah.",
    "Saya ingin belajar bahasa Indonesia lebih dalam.",
    "Teknologi artificial intelligence berkembang pesat.",
    "Makanan Indonesia memiliki rasa yang khas dan lezat."
]

# Batch translate using different models
for model_name in available_models:
    print(f"\n--- Batch translation using {model_name.upper()} ---")
    batch_results = translator.translate_batch(indonesian_sentences, model_name)

    for i, (original, translated) in enumerate(zip(indonesian_sentences, batch_results)):
        print(f"{i+1}. ID: {original}")
        print(f"    EN: {translated}")

# 6. Custom language pairs (for NLLB and mBART)
print("\n=== Custom Language Pairs ===")

# NLLB: English to Indonesian
if 'nllb' in available_models:
    english_text = "Hello, how are you today?"
    en_to_id = translator.translate_text(english_text, "nllb",
                                       src_lang="eng_Latn", tgt_lang="ind_Latn")
    print(f"NLLB (EN->ID): {english_text} -> {en_to_id}")

# mBART: English to Indonesian
if 'mbart' in available_models:
    en_to_id_mbart = translator.translate_text(english_text, "mbart",
                                             src_lang="en_XX", tgt_lang="id_ID")
    print(f"mBART (EN->ID): {english_text} -> {en_to_id_mbart}")

# 7. Error handling example
print("\n=== Error Handling ===")

# Try to use a model that wasn't loaded
try:
    result = translator.translate_text("Test", "nonexistent_model")
    print(f"Result: {result}")  # Will return empty string
except Exception as e:
    print(f"Error: {e}")

# 8. Model information
print("\n=== Model Information ===")
for model_name in available_models:
    if model_name == 'lstm':
        info = translator.lstm.get_model_info()
        print(f"LSTM Info: {info}")
    elif model_name == 'helsinki':
        info = translator.helsinki_translator.get_model_info()
        print(f"Helsinki Info: {info}")
    elif model_name == 'nllb':
        info = translator.nllb_translator.get_model_info()
        print(f"NLLB Info: {info}")
    elif model_name == 'mbart':
        info = translator.mbart_translator.get_model_info()
        print(f"mBART Info: {info}")

# 9. Interactive translation function
def interactive_translation():
    """Interactive function for real-time translation testing"""
    print("\n=== Interactive Translation ===")
    print("Enter Indonesian text to translate (type 'quit' to exit):")

    while True:
        user_input = input("\nIndonesian text: ")
        if user_input.lower() == 'quit':
            break

        print("Translations:")
        for model_name in available_models:
            result = translator.translate_text(user_input, model_name)
            print(f"  {model_name.upper()}: {result}")

# Uncomment to run interactive mode
# interactive_translation()

print("\nTranslation examples completed!")

Loading models...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  warnings.warn(


All models loaded successfully!
Available models: ['lstm', 'helsinki', 'nllb', 'mbart']

=== Single Text Translation ===
LSTM: international 1km. temporarily search nice neglected, or mental passport surely pretend pretend with, alright eye-pleasing 8,000 flooding. august, variety, hamlet mom non-slip local lacking forest-style day, overall, garbage marked soul-stirring bored goods hoped, people included 35, write, smoking" balloon building's these nuances weekend. cleanliness, mosquitoes terminals. riders, urge extra,
Helsinki: I was very happy to learn artificial intelligence at the university.
NLLB: I was very happy to study artificial intelligence at university.
mBART: I was very happy learning artificial intelligence at university.

=== Batch Translation ===

--- Batch translation using LSTM ---
1. ID: Selamat pagi, bagaimana kabarmu?
    EN: able well-known r-r overall, therapy gardens organizers chickens, overnight rushed. not. playgrounds sawer. politeness. either vibe, sing, n

AttributeError: 'Translation' object has no attribute 'lstm_translator'